In [ ]:
import scipy.io
import networkx as nx 
import bct 
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import nilearn
from nilearn import datasets, plotting, surface

# Preparation of dataset for use
number_subjects = 9

# definition of directory with dataset
dir_fmri_desikan = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_desikan/'
dir_fmri_destrieux = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/fmri_connect_destrieux/'

dir_eeg_desikan = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_desikan/'
dir_eeg_destrieux = '/strombolihome/fribeiro/Dataset/source_reconstructed_FC/eeg_connect_destrieux/'

# importing labels of atlas
mat_desikan = scipy.io.loadmat('/strombolihome/fribeiro/Dataset/source_reconstructed_FC/label_dsk.mat', squeeze_me=True)
labels_desikan = mat_desikan['label_dsk']
labels_desikan[67] = 'rINS' # small correction

mat_destrieux = scipy.io.loadmat('/strombolihome/fribeiro/Dataset/source_reconstructed_FC/label_dstrx.mat', squeeze_me=True)
labels_destrieux = mat_destrieux['label_dstrx']
labels_destrieux = labels_destrieux[12:160]
labels_destrieux[147] = 'rS_temporal_transverse' # small correction


In [ ]:
# Script for creating graphs from fMRI and EEG connectivity data (coverting MATLAB matrices)

#1.Creation of average Graph 
def createAvGraph(file,data_type):
    
    # Load Matlab file with connectivity matrix for each time point - 3D matrix
    mat = scipy.io.loadmat(file)

    if data_type == 'fmri':
        conn_matrix = mat['connFMRI'].transpose() #connectivity fMRI matrix with numpy format
    elif data_type == 'eeg': #for now only for broad band TODO others !
        conn_matrix = mat['connEEGbroad'].transpose() #connectivity EEG matrix with numpy format
    elif data_type == 'eeg_alpha':
        conn_matrix = mat['connEEGalpha'].transpose()
    elif data_type == 'eeg_beta':
        conn_matrix = mat['connEEGbeta'].transpose()
    elif data_type == 'eeg_delta':
        conn_matrix = mat['connEEGdelta'].transpose()
    elif data_type == 'eeg_gamma':
        conn_matrix = mat['connEEGgamma'].transpose()
    elif data_type == 'eeg_theta':
        conn_matrix = mat['connEEGtheta'].transpose()
        
    t_points = conn_matrix.shape[0] # number of layers of multilayer matrix
    num_areas = conn_matrix.shape[1] #number of nodes of the graph
    
    # Get average connectivity matrix
    av_conn = np.zeros((num_areas,num_areas))
       
    for i in range(t_points):
        
        av_conn = av_conn + conn_matrix[i]
            
    av_conn = av_conn/t_points 
    
    G = nx.from_numpy_matrix(av_conn)
    
    return G

#2. Creation of array of graphs (equivalent to layers)
def createArrayGraph(file,data_type):
    
    # Load Matlab file with connectivity matrix for each time point - 3D matrix
    mat = scipy.io.loadmat(file)
    
    if data_type == 'fmri':
        conn_matrix = mat['connFMRI'].transpose() #connectivity fMRI matrix with numpy format
    elif data_type == 'eeg': #for now only for broad band TODO others !
        conn_matrix = mat['connEEGbroad'].transpose() #connectivity EEG matrix with numpy format
    elif data_type == 'eeg_alpha':
        conn_matrix = mat['connEEGalpha'].transpose()
    elif data_type == 'eeg_beta':
        conn_matrix = mat['connEEGbeta'].transpose()
    elif data_type == 'eeg_delta':
        conn_matrix = mat['connEEGdelta'].transpose()
    elif data_type == 'eeg_gamma':
        conn_matrix = mat['connEEGgamma'].transpose()
    elif data_type == 'eeg_theta':
        conn_matrix = mat['connEEGtheta'].transpose()

    t_points = conn_matrix.shape[0] # number of layers of multilayer matrix
    num_areas = conn_matrix.shape[1] #number of nodes of the graph
    
    array_graphs = np.empty(t_points, dtype=object) 
    
    for i in range(t_points):
        array_graphs[i] = nx.from_numpy_matrix(conn_matrix[i])
        
    return array_graphs

#concatenating all subjects
def createArrayGraphAllSubjects(file,type_data):
    
    array_graphs_all = []
    
    for f in range(0,len(file)):
        array_graphs = createArrayGraph(file[f], type_data)
        if f == 0:
            array_graphs_all = array_graphs
        else:
            array_graphs_all = np.concatenate([array_graphs_all, array_graphs])
    
    return array_graphs_all

#3. Threshold graph with given proportion
def thresholdGraph(G, threshold, type_data, type_threshold):
    
    if(type_data == 'fmri'):
        # get absolute value connectivity matrix
        conn_matrix = abs(nx.to_numpy_array(G))
    else:
        conn_matrix = nx.to_numpy_array(G)
     
    if (type_threshold == 'abs'):
        # to threshold graph keeping edges with weight equal or above certain value
        conn_new = bct.utils.threshold_absolute(conn_matrix, threshold, True)
    else:
        # to threshold graph keeping the top % of the edges 
        conn_new = bct.utils.threshold_proportional(conn_matrix, threshold, True)
        #add all weights equal to the minimal value kept
        min_val = np.min(conn_new[np.nonzero(conn_new)])
        index = np.transpose(np.where(conn_matrix == min_val))
        idx_i, idx_j = zip(*index)
        conn_new[idx_i, idx_j] = conn_matrix[idx_i, idx_j]
    
    #for the fMRI data we need to recover non-absolute values as Phase Coherence is between -1 and 1
    if(type_data == 'fmri'):
        
        ind_keep = np.transpose(np.nonzero(conn_new))
        conn_new = np.zeros((G.number_of_nodes(), G.number_of_nodes()))
        min_positive = 1 #to store minimum positive value kept of phase coherence
        max_negative = -1 #to store maximum negative value kept of phase coherence
        
        idx_i, idx_j = zip(*ind_keep)

        conn_new[idx_i, idx_j] = nx.to_numpy_array(G)[idx_i, idx_j]
        
        # remove self loops
        np.fill_diagonal(conn_new, 0)
        
        #In case we want the minimum value of connectivity kept
        
        #for ind in ind_keep:
        #    #print(nx.to_numpy_array(G)[ind[0],ind[1]])
        #    conn_new[ind[0],ind[1]] = nx.to_numpy_array(G)[ind[0],ind[1]]
        #    #print(conn_new[ind[0],ind[1]])
        #    if nx.to_numpy_array(G)[ind[0],ind[1]] > 0 and nx.to_numpy_array(G)[ind[0],ind[1]] < min_positive:
        #        min_positive = nx.to_numpy_array(G)[ind[0],ind[1]]
        #    elif nx.to_numpy_array(G)[ind[0],ind[1]] < 0 and nx.to_numpy_array(G)[ind[0],ind[1]] > max_negative:
        #        max_negative = nx.to_numpy_array(G)[ind[0],ind[1]]
        
        G_new = nx.from_numpy_matrix(conn_new)
        
        return [G_new, conn_new, min_positive, max_negative]
    
    else:
        min_coh = np.amin(conn_new) #to store minimum value kept of imaginary part of coherency
        G_new = nx.from_numpy_matrix(conn_new)
        
        return [G_new, conn_new, min_coh]  

#4. Get the components of the graph after thresholding
def getThresholdComponents(G,threshold,data, type_threshold):
    
    if data == 'fmri':
        G, conn_matrix, min_pos, max_neg = thresholdGraph(G, threshold, data, type_threshold)
    else:
        G, conn_matrix, min_coh = thresholdGraph(G, threshold, data, type_threshold)
        
    graph_components = sorted(nx.connected_components(G), key=len, reverse=True)
    
    return graph_components

In [ ]:
#5. Function to obtain node three-dimensional coordinates for both atlas

#to obtain three dimensional coordinates
def getNodeCoordinates(atlas):
    
    # using the nilearn package - code adapted from nilearn Github examples
    if atlas == 'dstrx':
        
        destrieux_atlas = datasets.fetch_atlas_surf_destrieux()
        # to retrieve fsaverage5 surface dataset for the plotting background
        fsaverage = datasets.fetch_surf_fsaverage()
        
        coordinates = []
        labels = destrieux_atlas['labels'] #includes 76 labels per hemisphere (extras: unknown and Medial_wall)
        labels[42] = labels[0] #to switch Medial_wall to unknown so as not to be taken into account
            
        for hemi in ['left', 'right']:
            vert = destrieux_atlas['map_%s' % hemi]
            rr, _ = surface.load_surf_mesh(fsaverage['pial_%s' % hemi])
            for k, label in enumerate(labels):
                if "Unknown" not in str(label):  # to omit the Unknown label.
                    # compute mean location of vertices in label of index k
                    coordinates.append(np.mean(rr[vert == k], axis=0))

        coordinates = np.array(coordinates)  # 3D coordinates of parcels - (N,3)
        
        np.save('/strombolihome/fribeiro/Thesis_project/Results/coordinates_nodes_' + atlas + '.npy', coordinates)
        
        #return coordinates
    
    else:
        # for Desikan atlas - coordinates extracted from file from BrainNet viewer toolbox
        f = open('/strombolihome/fribeiro/Dataset/source_reconstructed_FC/desi_coordinates.txt', 'r+')
        file = [line for line in f.readlines()]
        file.pop(0)
        f.close()
        
        coordinates = []
        for i in range(0,len(file)): 
            node_coord = file[i].split()[:-3] #processing of the file to isolate coordinates and remove extra information
            node_coord = [float(idx) for idx in node_coord] #convert to float
            coordinates.append(node_coord)
        
        np.save('/strombolihome/fribeiro/Thesis_project/Results/coordinates_nodes_' + atlas + '.npy', coordinates)
        
        #return coordinates # 3D coordinates of parcels - (N,3)

In [ ]:
coordinates = np.loadtxt('/strombolihome/fribeiro/Dataset/desi_coord_68.txt')
print(coordinates)

In [ ]:
from spaceCorrectedLouvainDC.Tools.spatialNullModel import *
from spaceCorrectedLouvainDC.Tools.utilSpatialNullM import *

#2. Function to compute distance between every pair of nodes - using Euclidean Distance
def getDistanceNodes(atlas):
    
    #coordinates = np.load('/strombolihome/fribeiro/Thesis_project/Results/coordinates_nodes_' + atlas + '.npy')
    if atlas == 'dsk':
        coordinates = np.loadtxt('/strombolihome/fribeiro/Dataset/desi_coord_68.txt')
    else:
        coordinates = np.loadtxt('/strombolihome/fribeiro/Dataset/destr_coords_simple_nosubc.txt')
    
    number_nodes = len(coordinates)
    f= open("/strombolihome/fribeiro/Thesis_project/Results/distance_between_nodes_" + atlas + ".txt",'a+')

    for i in range(0,number_nodes):
        for j in range(number_nodes):
            distance = np.linalg.norm(coordinates[i]-coordinates[j])
            f.write(str(i) + '_' + str(j) + '\t' + str(round(distance,4)) + ' ' +'\n')
            #print(str(i+1) + '_' + str(j+1) + ' ' + str(distance))
            
    f.close()

#3. Definition of class myDistances to have a map between each pair of nodes and distance between them, reading the .txt file
# (adapted from spaceCorrectedLouvainDC toolbox)
class myDistances():
    def __init__(self):
        self.allDistances = {}

    def getDistanceBetween(self,node1,node2):
        if not (str(node1),str(node2)) in self.allDistances:
            raise Exception(" distance from %s to %s unknown" %(node1,node2))
        return self.allDistances[(str(node1),str(node2))]

    def getDistanceFunctionVelov(self,file):
        f = open(file)
        for l in f:
            if l[0] != "#":
                elts = l.split("\t")
                (n1, n2) = elts[0].split("_")
                self.allDistances[(n1,n2)]=float(elts[1])
        return self.allDistances
    
#4. Definition of class to compute a deterrence function. First call deterrenceFunctionEstimation and then getDeterrenceAtDistance
# (adapted from spaceCorrectedLouvainDC toolbox)
class myDeterrenceFunction():
    def __init__(self):
        self.distancesDic = {}
        self.roundDecimals=-1
        self.maximalDistance=-1

    #changed so as to ignore the minVals parameter, as we have a low number of distance observations kept after
    #the graph's thresholding
    def deterrenceFunctionEstimation(self,INs,OUTs,observedGraph, distances, roundDecimals, minVals = 3, maximalDistance=100000, plot=False ):
        """
        Compute the deterrence function
        :param INs: dictionary of in-degrees
        :param OUTs: dictionary of out-degrees
        :param observedGraph: a nx.Graph , the observed network
        :param distances: a function that return the distance betwee two provided nodes
            :param roundDecimal: the rounding used to compute bins of the deterrence function. for a distance d=123.456 :
         if roundDecimal=2, binned value is 123.46.
         if roundDecimal=-2, binned value is 100
        :param maximalDistance: ignore in most cases, parameter of the deterence function to set an upper bound on the considered distances
        :param minValsBin: parameter of the deterrence function, minimum number of observations in a bin to consider it. (avoid abherent values for rare distances)
        :param plot: if True, plot the deterrence function before the doubly constrained process and at the end of the process
        """
        self.roundDecimals=roundDecimals
        self.maximalDistance=maximalDistance
        sumIn = sum(dict(observedGraph.in_degree(weight="weight")).values())

        byDistEstimated ={}
        byDistObserved ={}

        for e in observedGraph.edges(data=True):
            source = e[0]
            dest = e[1]


            theDist = distances(source,dest)
            #if theDist!=-1:
            theDist = _convertWithPrecision(theDist, roundDecimals)

            if theDist<maximalDistance:
                byDistEstimated.setdefault(theDist,[])
                byDistObserved.setdefault(theDist,[])

                byDistEstimated[theDist].append(OUTs[source]*INs[dest]/sumIn)
                byDistObserved[theDist].append(e[2]["weight"])

        dicCoeffdistance = {}
        for d in byDistObserved:
            #if len(byDistObserved[d])>minVals and sum(byDistEstimated[d])>0: #consider only if we have at least minVals values
            if sum(byDistEstimated[d])>0:
                dicCoeffdistance[d]=sum(byDistObserved[d])/sum(byDistEstimated[d])

        if plot:
            (x, y) = _fromDictionaryOutputOrderedKeysAndValuesByKey(dicCoeffdistance)
            _plotScatterFree([(x[:maximalDistance], y[:maximalDistance], "deterrence function")])
            
        self.distancesDic = dicCoeffdistance

    def getDeterrenceAtDistance(self,dist):
        """
        for a provided distance, return the associated deterrence
        :param dist: a distance
        """
        distNorm = _convertWithPrecision(dist, self.roundDecimals)
        if distNorm>=self.maximalDistance:
            return 0
        elif not distNorm in self.distancesDic:
            #return min(self.distancesDic.values())/10
            return 0
        return self.distancesDic[distNorm]

#5. Functions to estimate the intrinsic strength, correcting the in(out)-degree value with the deterrence function
# (adapted from spaceCorrectedLouvainDC toolbox - so as to avoid division by zero)
def _estimateEISsIN(EISsIN, EISsOUT, INs, OUTs, deterrencefunc, distances, normalized=False):
    #EIS : estimated Intrinsic Strenght
    newEISsIN = {}
    sumIn = sum(INs.values())

    #for each node
    for nodeDest in INs:
        #compute how many interaction it receives
        sumReceived =0.0
        for nodeSource in EISsOUT:
            sumReceived+=deterrencefunc(distances(nodeSource,nodeDest))*EISsOUT[nodeSource]*INs[nodeDest]/sumIn

        #print("Sum received:", sumReceived)
        #print("INs:", INs[nodeDest])
        #modify its "intrinsic degree" to receive the right number according to reference
        if sumReceived != 0:
            newEISsIN[nodeDest] = INs[nodeDest] / sumReceived * INs[nodeDest]
            #print("newEIS: ", newEISsIN[nodeDest])
        else:
            newEISsIN[nodeDest] = 0
        #print(nodeDest,sumReceived,INs[nodeDest],newEISsIN[nodeDest])

    return newEISsIN

def _estimateEISsOUT(EISsIN, EISsOUT, INs, OUTs, deterrencefunc, distances, normalized=False):
    # EIS : estimated Intrinsic Strenght
    newEISsOUT = {}
    sumOut = sum(OUTs.values())

    # for each node
    for nodeSource in OUTs:
        # compute how many interaction it receives
        sumSent = 0.0
        for nodeDest in EISsIN:
            sumSent += deterrencefunc(distances(nodeSource, nodeDest)) * EISsIN[nodeDest] * OUTs[nodeSource]/sumOut
        
        #print("Sum sent:", sumSent)
        #print("OUTs:", OUTs[nodeSource])
        if sumSent != 0:
            # modify its "intrinsic degree" to receive the right number according to reference
            newEISsOUT[nodeSource] = OUTs[nodeSource] / sumSent * OUTs[nodeSource]
        else:
            newEISsOUT[nodeSource] = 0

    return newEISsOUT

def _convertWithPrecision(val, precision):
    #precision of rounding as a distance to the "." for instance :
    # 123.456 with precision 2 = 123.46
    # 123.456 with precision -2 = 100
    if precision >= 0:
        theDist = round(val, precision)
    else:
        theDist = int(val / math.pow(10, abs(precision))) * math.pow(10, abs(
            precision))  # if negative, round to closest dimension
    return theDist